In [1]:
!pip install happybase

In [2]:
import pandas as pd
import happybase

In [3]:
df = pd.read_csv("/home/jovyan/data/movie_success_rate.csv")

In [4]:
conn = happybase.Connection(
    host="hbase",
    port=9090,
    timeout=20000
)
conn.open()

if b'movie_success_rate_raw' not in conn.tables():
    conn.create_table(
        'movie_success_rate_raw',
        {'cf': dict()}
    )

table = conn.table('movie_success_rate_raw')

In [5]:
CHUNK_SIZE = 100

for start in range(0, len(df), CHUNK_SIZE):
    end = start + CHUNK_SIZE
    chunk = df.iloc[start:end]

    batch = table.batch() 

    for idx, row in chunk.iterrows():
        if pd.isna(row["Title"]):
            continue

        rowkey = f"{row['Title']}_{row['Year']}_{idx}".encode()

        batch.put(rowkey, {
            b'cf:title': str(row['Title']).encode(),
            b'cf:year': str(row['Year']).encode(),
            b'cf:rating': str(row['Rating']).encode(),
            b'cf:votes': str(row['Votes']).encode(),
            b'cf:revenue': str(row['Revenue (Millions)']).encode(),
            b'cf:metascore': str(row['Metascore']).encode(),
            b'cf:success': str(row['Success']).encode(),
        })

    batch.send()
    print(f"Inserted rows {start} to {end}")

Inserted rows 0 to 100
Inserted rows 100 to 200
Inserted rows 200 to 300
Inserted rows 300 to 400
Inserted rows 400 to 500
Inserted rows 500 to 600
Inserted rows 600 to 700
Inserted rows 700 to 800
Inserted rows 800 to 900
